# 2장 2강: 분산분석(ANOVA)과 사후 검정 이론 — 실습문제

## 실습 목표

- 세 집단 이상의 평균을 일원배치 분산분석으로 비교할 수 있다.
- F통계량과 p-value를 이용하여 전체 집단 차이를 판단할 수 있다.
- 집단 간·집단 내 변동으로 ANOVA 표를 구성하고 해석할 수 있다.
- ANOVA가 유의할 때 Tukey HSD 사후 검정을 수행할 수 있다.
- 유의한 집단 쌍과 평균 차이의 크기를 근거로 차이 구조를 설명할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing(1).csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `KitchenQual` | 주방 품질 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> 표본은 지정된 `random_state`로 추출하여 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df = pd.read_csv("ames_housing.csv")
df.head()

,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. One-way ANOVA로 전체 평균 차이 확인

### 문제 1-1. 전반적인 품질 점수 5·6·7 집단 비교

#### 문제 설명

주택의 전반적인 품질 점수가 5점, 6점, 7점인 세 집단의 평균 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출하고 일원배치 분산분석을 수행하세요.

#### 요구사항

1. `OverallQual`이 5, 6, 7인 각 집단의 `SalePrice`에서 `n=20`, `random_state=5`로 표본을 추출하세요.
2. 각 집단의 표본 수, 평균, 표준편차를 출력하세요.
3. 세 집단이 서로 다른 주택으로 구성된 독립집단임을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 세 집단에 Levene 등분산 검정을 수행하세요.
6. 다음 가설을 작성하세요.
   - H₀: 세 집단의 모집단 평균 판매가격은 모두 같다.
   - H₁: 적어도 한 집단의 모집단 평균 판매가격은 다르다.
7. `stats.f_oneway()`로 일원배치 ANOVA를 수행하세요.
8. F통계량과 p-value를 출력하고 전체 차이 유무를 판단하세요.
9. ANOVA 결과만으로 어느 집단끼리 다른지 알 수 있는지 설명하세요.

#### 해석 질문

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검 결과
- 가설 설정
- F통계량과 p-value
- 전체 차이 판단
- 사후 검정 필요성 설명
- Q1~Q4 답변

In [2]:
# 필수 1 코드를 작성하세요.
alpha = 0.05

quality_5 = df.loc[df["OverallQual"] == 5, "SalePrice"].dropna().sample(n=20, random_state=5)
quality_6 = df.loc[df["OverallQual"] == 6, "SalePrice"].dropna().sample(n=20, random_state=5)
quality_7 = df.loc[df["OverallQual"] == 7, "SalePrice"].dropna().sample(n=20, random_state=5)

groups = {"5": quality_5, "6": quality_6, "7": quality_7}

for name, group in groups.items():
    normal_p = stats.shapiro(group).pvalue
    print(f"품질 {name}: n={len(group)}, 평균={group.mean():,.2f}, 표준편차={group.std(ddof=1):,.2f}, 정규성 p={normal_p:.4f}")

levene_result = stats.levene(quality_5, quality_6, quality_7)

# H0: 세 집단의 모집단 평균 판매가격은 모두 같다.
# H1: 적어도 한 집단의 모집단 평균 판매가격은 다르다.

anova_result = stats.f_oneway(quality_5, quality_6, quality_7)

print("독립성: 서로 다른 주택으로 구성된 독립집단")
print(f"등분산성 p-value: {levene_result.pvalue:.4f}")
print(f"F통계량: {anova_result.statistic:.4f}")
print(f"p-value: {anova_result.pvalue:.4f}")
print(f"전체 판단: {'적어도 한 집단의 평균이 다름' if anova_result.pvalue < alpha else '세 집단의 평균 차이를 확인하지 못함'}")
print("ANOVA 결과만으로는 어느 집단끼리 다른지 알 수 없음")


품질 5: n=20, 평균=130,605.00, 표준편차=24,937.11, 정규성 p=0.7760
품질 6: n=20, 평균=167,826.60, 표준편차=41,944.55, 정규성 p=0.4096
품질 7: n=20, 평균=217,593.60, 표준편차=48,298.39, 정규성 p=0.1290
독립성: 서로 다른 주택으로 구성된 독립집단
등분산성 p-value: 0.0792
F통계량: 24.2456
p-value: 0.0000
전체 판단: 적어도 한 집단의 평균이 다름
ANOVA 결과만으로는 어느 집단끼리 다른지 알 수 없음


### 필수 1 답변 작성란

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
 - 검정을 반복할수록 전체 분석에서 한 번 이상 제1종 오류가 발생 확률이 0.05보다 커지는 다중 비교 문제가 발생한다.

**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
 - 집단 간 평균 차이를 나타내는 집단 간 변동을 같은 집단 내부의 개인차인 집단 내 변동으로 나눈 비율

**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요? 
 - ㅇㅇ 적어도 한 집단의 모집단 평균 판매 가격은 다르다.

**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?
 - ㄴㄴ 구체적인 집단 쌍은 Turkey HSD와 같은 사후 검정으로 확인해야 한다.

---

## 필수 2. ANOVA 표 구성과 Tukey HSD 사후 검정

### 문제 2-1. 품질 점수별 차이 구조 확인

#### 문제 설명

필수 1의 세 집단을 이용하여 ANOVA 표를 직접 구성하고, 전체 차이가 유의한 경우 Tukey HSD 사후 검정을 수행해 어느 품질 점수 집단끼리 차이가 있는지 확인하세요.

#### 요구사항

1. 필수 1의 세 집단을 하나의 `anova_df` 데이터프레임으로 결합하세요.
2. 전체 평균을 계산하세요.
3. 다음 값을 계산하여 ANOVA 표를 만드세요.
   - 집단 간 제곱합 `SS_between`
   - 집단 내 제곱합 `SS_within`
   - 집단 간·집단 내 자유도
   - 평균제곱 `MS_between`, `MS_within`
   - F통계량
4. 직접 계산한 F통계량이 `stats.f_oneway()` 결과와 일치하는지 확인하세요.
5. ANOVA p-value가 0.05보다 작을 때만 `pairwise_tukeyhsd()`를 실행하세요.
6. Tukey 결과에서 `reject=True`인 집단 쌍을 확인하세요.
7. 각 집단 평균을 이용하여 집단 쌍별 평균 차이를 계산하세요.
8. 어느 집단 쌍이 유의하며 차이가 가장 큰 집단 쌍은 무엇인지 해석하세요.

#### 해석 질문

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- ANOVA 표
- F통계량 대조 결과
- Tukey HSD 결과
- 유의한 집단 쌍
- 집단 쌍별 평균 차이
- Q1~Q4 답변

In [5]:
# 필수 2 코드를 작성하세요.
anova_df = pd.DataFrame({
    "SalePrice": pd.concat([quality_5, quality_6, quality_7], ignore_index=True),
    "QualityGroup": ["5"] * len(quality_5) + ["6"] * len(quality_6) + ["7"] * len(quality_7)
})

grand_mean = anova_df["SalePrice"].mean()
group_means = anova_df.groupby("QualityGroup")["SalePrice"].mean()
group_sizes = anova_df.groupby("QualityGroup")["SalePrice"].size()

ss_between = sum(
    group_sizes[g] * (group_means[g] - grand_mean) ** 2 for g in group_means.index
)

ss_within = sum(
    ((group["SalePrice"] - group["SalePrice"].mean()) ** 2).sum()
    for _, group in anova_df.groupby("QualityGroup")
)

k = anova_df["QualityGroup"].nunique()
n = len(anova_df)

df_between = k - 1
df_within = n - k

ms_between = ss_between / df_between
ms_within = ss_within / df_within
manual_f = ms_between / ms_within

anova_table = pd.DataFrame({
    "SS": [ss_between, ss_within, ss_between + ss_within],
    "df": [df_between, df_within, n - 1],
    "MS": [ms_between, ms_within, np.nan],
    "F": [manual_f, np.nan, np.nan]
}, index=["집단 간", "집단 내", "전체"])

print("ANOVA 표")
print(anova_table)
print(f"직접 계산한 F: {manual_f:.4f}")
print(f"scipy F: {anova_result.statistic:.4f}")
print(f"F통계량 일치 여부: {np.isclose(manual_f, anova_result.statistic)}")

if anova_result.pvalue <= alpha:
    tukey_result = pairwise_tukeyhsd(
        endog=anova_df["SalePrice"],
        groups=anova_df["QualityGroup"],
        alpha=alpha
    )
    print(tukey_result)

print("\n집단 평균")
print(group_means)
print(f"5-6 평균 차이: {abs(group_means['5'] - group_means['6']):,.2f}")
print(f"5-7 평균 차이: {abs(group_means['5'] - group_means['7']):,.2f}")
print(f"6-7 평균 차이: {abs(group_means['6'] - group_means['7']):,.2f}")

ANOVA 표
                SS  df            MS          F
집단 간  7.619479e+10   2  3.809739e+10  24.245575
집단 내  8.956486e+10  57  1.571313e+09        NaN
전체    1.657596e+11  59           NaN        NaN
직접 계산한 F: 24.2456
scipy F: 24.2456
F통계량 일치 여부: True
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower       upper    reject
-----------------------------------------------------------
     5      6  37221.6  0.012  7056.6586  67386.5414   True
     5      7  86988.6    0.0 56823.6586 117153.5414   True
     6      7  49767.0 0.0006 19602.0586  79931.9414   True
-----------------------------------------------------------

집단 평균
QualityGroup
5    130605.0
6    167826.6
7    217593.6
Name: SalePrice, dtype: float64
5-6 평균 차이: 37,221.60
5-7 평균 차이: 86,988.60
6-7 평균 차이: 49,767.00


### 필수 2 답변 작성란

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
 - SS_between은 집단 평균들이 전쳋 평균에서 벗어난 집단 간 변동값
 - SS_within은 각 관측값이 소속 집단 평균에서 벗어난 집단 내 변동

**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요? 
 - 다중비교 오류를 조정한 뒤에도 해당 두 집단의 평균이 같다는 귀무가설을 기각하겠다는 의미

**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
 - 5-6, 5-7, 6-7의 모든 집단 쌍에서 유의한 차이가 확인된다.

**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
 - 품질 5점과 7점 집단이며 평균 차이는 약 86,988 달러이다.

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질 `Ex`·`Gd`·`TA` 집단 비교

#### 문제 설명

주방 품질이 `Ex`, `Gd`, `TA`인 세 집단의 평균 판매가격을 비교합니다. 각 집단에서 20개씩 표본을 추출한 뒤 ANOVA와 Tukey HSD를 순서대로 적용하세요.

> 필수 문제에서 학습한 전체 검정→사후 검정 절차를 새로운 집단 변수에 적용하는 과제입니다.

#### 요구사항

1. `KitchenQual`이 `Ex`, `Gd`, `TA`인 각 집단의 `SalePrice`에서 `n=20`, `random_state=18`로 표본을 추출하세요.
2. 세 집단의 표본 수와 평균을 출력하세요.
3. 정규성과 등분산성을 확인하세요.
4. 일원배치 ANOVA를 수행하고 F통계량과 p-value를 출력하세요.
5. ANOVA가 유의한 경우에만 Tukey HSD 사후 검정을 수행하세요.
6. Tukey 결과에서 유의한 집단 쌍을 확인하세요.
7. 각 집단 쌍의 평균 판매가격 차이를 계산하세요.
8. 어느 집단 쌍의 차이가 가장 큰지 포함하여 주방 품질별 차이 구조를 해석하세요.

#### 해석 질문

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검
- ANOVA 결과
- Tukey HSD 결과
- 유의한 집단 쌍과 평균 차이
- 최종 해석
- Q1~Q4 답변

In [2]:
# 과제 코드를 작성하세요.
alpha = 0.05

kitchen_ex = df.loc[df["KitchenQual"] == "Ex", "SalePrice"].dropna().sample(n=20, random_state=18)
kitchen_gd = df.loc[df["KitchenQual"] == "Gd", "SalePrice"].dropna().sample(n=20, random_state=18)
kitchen_ta = df.loc[df["KitchenQual"] == "TA", "SalePrice"].dropna().sample(n=20, random_state=18)

groups = {
    "Ex": kitchen_ex,
    "Gd": kitchen_gd,
    "TA": kitchen_ta
}

for name, group in groups.items():
    normal_stat, normal_p = stats.shapiro(group)
    print(
        f"{name}: n={len(group)}, 평균={group.mean():,.2f}, "
        f"정규성 p-value={normal_p:.4f}"
    )

levene_stat, levene_p = stats.levene(
    kitchen_ex,
    kitchen_gd,
    kitchen_ta
)

print(f"등분산성 p-value: {levene_p:.4f}")

f_stat, p_value = stats.f_oneway(
    kitchen_ex,
    kitchen_gd,
    kitchen_ta
)

print(f"F통계량: {f_stat:.4f}")
print(f"p-value: {p_value:.4f}")

anova_df = pd.DataFrame({
    "SalePrice": pd.concat(
        [kitchen_ex, kitchen_gd, kitchen_ta],
        ignore_index=True
    ),
    "KitchenGroup":
        ["Ex"] * len(kitchen_ex)
        + ["Gd"] * len(kitchen_gd)
        + ["TA"] * len(kitchen_ta)
})

if p_value <= alpha:
    tukey_result = pairwise_tukeyhsd(
        endog=anova_df["SalePrice"],
        groups=anova_df["KitchenGroup"],
        alpha=alpha
    )
    print(tukey_result)

group_means = anova_df.groupby("KitchenGroup")["SalePrice"].mean()

print("\n집단 평균")
print(group_means)

print(f"Ex-Gd 평균 차이: {abs(group_means['Ex'] - group_means['Gd']):,.2f}")
print(f"Ex-TA 평균 차이: {abs(group_means['Ex'] - group_means['TA']):,.2f}")
print(f"Gd-TA 평균 차이: {abs(group_means['Gd'] - group_means['TA']):,.2f}")

Ex: n=20, 평균=313,983.05, 정규성 p-value=0.6875
Gd: n=20, 평균=188,835.00, 정규성 p-value=0.8024
TA: n=20, 평균=137,486.60, 정규성 p-value=0.3973
등분산성 p-value: 0.0866
F통계량: 54.8000
p-value: 0.0000
      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
group1 group2  meandiff  p-adj     lower        upper     reject
----------------------------------------------------------------
    Ex     Gd -125148.05    0.0 -166883.2108  -83412.8892   True
    Ex     TA -176496.45    0.0 -218231.6108 -134761.2892   True
    Gd     TA   -51348.4 0.0122  -93083.5608   -9613.2392   True
----------------------------------------------------------------

집단 평균
KitchenGroup
Ex    313983.05
Gd    188835.00
TA    137486.60
Name: SalePrice, dtype: float64
Ex-Gd 평균 차이: 125,148.05
Ex-TA 평균 차이: 176,496.45
Gd-TA 평균 차이: 51,348.40


### 과제 답변 작성란

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?
  - p-value가 0.05보다 작으므로 세 집단의 평균에는 유의한 차이가 있다고 해석된다.

**Q2.** 사후 검정은 어떤 조건에서 수행하나요?
  - ANOVA에서 유의한 결과가 나온 경우 구체적으로 어떤 집단끼리 다른지 확인하기 위해 수행한다.

**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?
  - 세 집단 쌍 모두 유의한 차이가 확인되었다.

**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
  - Ex-TA 집단이 176,496 달러로 가장 크다.

---

## 실습 마무리

1. 세 집단 이상을 t검정으로 반복 비교하면 왜 제1종 오류가 커지나요?
 - 각 검정마다 위양성 가능성이 있기 때문에 비교 횟수가 늘어날수록 전체 분석에서 한 번 이상 잘못 기각할 확률이 누적이다.

2. ANOVA의 귀무가설과 대립가설은 무엇인가요?
 - 귀무가설은 모든 집단의 모집단과 평균이 같다
 - 대립가설은 적어도 한 집단의 평균이 다르다라는 것

3. F통계량이 크다는 것은 무엇을 의미하나요?
 - 집단 내 변동에 비해 집단 간 평균 차이로 설명되는 변동이 상대적으로 크다는 의미

4. ANOVA가 유의하더라도 사후 검정이 필요한 이유는 무엇인가요?
 - ANOVA는 적어도 한 집단이 다르다는 사실만 알려 주며 구체적인 집단 쌍은 알려 주지 않음

5. Tukey HSD 결과에서 어떤 항목을 확인해야 하나요?
 - 비교한 집단 쌍, 평균 차이, 조정된 p-value, 신뢰구간, reject 여부